# J0 — la chaîne complète en six lignes

**IAA1302.1 — Apprentissage Automatique · ESGC-VAK 2026-2027**

Ce notebook fait **tout le trajet** : lire les données, séparer la cible, entraîner, prédire,
écrire un fichier de soumission, le déposer. Aucune explication de ce qui se passe à
l'intérieur : on regarde **ce qui entre** et **ce qui sort**.

## Les trois règles du dépôt

1. **`Save & Run All`** (version sauvegardée, exécutée de bout en bout, état propre).
2. **Privé** : le notebook n'est visible que de vous et de l'enseignant.
3. **CPU, moins de 10 minutes.** Le GPU est un bonus, jamais un prérequis.

## À déposer à la fin de la séance

- le **lien** de ce notebook (privé, exécuté) ;
- votre **tableau de 3 scores** : baseline naïve, score interne, score public.

---

In [1]:
# 1 — les outils
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

In [2]:
# 2 — où sont les fichiers (le notebook marche sur Kaggle ET en local)
# On CHERCHE le dossier des données au lieu de le supposer : sur Kaggle, une compétition
# monte ses fichiers sous /kaggle/input/<identifiant>, et un identifiant recopié à la main
# se périme dès qu'une compétition est recréée.
CANDIDATS = ["/kaggle/input/prevision-non-presentation-rendez-vous-2026-2027"]
CANDIDATS += sorted(glob.glob("/kaggle/input/*"))
CANDIDATS += ["sortie", os.path.join("..", "sortie"),
              os.path.join("..", "Donnees_competition", "sortie")]
DOSSIER = next((d for d in CANDIDATS if os.path.exists(os.path.join(d, "train.csv"))), None)
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"
CHEMIN_TRAIN = os.path.join(DOSSIER, "train.csv")
CHEMIN_TEST = os.path.join(DOSSIER, "test.csv")
print("train :", CHEMIN_TRAIN, "|", "test :", CHEMIN_TEST)

train : sortie\train.csv | test : sortie\test.csv


In [3]:
# 3 — lire, séparer la cible, garder de côté
train = pd.read_csv(CHEMIN_TRAIN, low_memory=False)
test = pd.read_csv(CHEMIN_TEST, low_memory=False)
ID = "ligne_id"
CIBLE = "patient_absent"
# Ce que ce notebook met de côté — volontairement, et sans l'expliquer aujourd'hui :
#   · les identifiants et les colonnes connues APRÈS l'issue (fuite) ;
#   · les dates brutes : une date n'est pas un nombre — S9 montrera comment la transformer ;
#   · `taille_cm` et `date_naissance`, saisies en TEXTE — S2 montrera comment les reprendre.
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance"]
X = train.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in train.columns])
y = train[CIBLE]
# Les colonnes non numériques sont déclarées catégorielles. Détection par `is_numeric_dtype`
# et non par `dtype == "object"` : pandas 3 a un dtype `str` dédié, pandas 2 un `object`.
CAT = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
# Une colonne à très forte cardinalité n'est pas une catégorielle : les modèles d'arbres
# acceptent les catégorielles natives jusqu'à 255 modalités. Au-delà, il faut un autre
# traitement (regroupement, encodage) — S10. On l'écarte ici, et on dit pourquoi.
TROP_FIN = [c for c in CAT if X[c].nunique() > 255]
if TROP_FIN:
    print("écartée(s), cardinalité > 255 :", [(c, int(X[c].nunique())) for c in TROP_FIN])
X = X.drop(columns=TROP_FIN)
CAT = [c for c in CAT if c not in TROP_FIN]
for c in CAT:
    X[c] = X[c].fillna("").astype("category")
print(train.shape, test.shape, "| cible :", round(100 * y.mean(), 2), "% de positifs")
print("colonnes catégorielles :", CAT)

écartée(s), cardinalité > 255 : [('tournee_jour', 9000)]
(149894, 39) (63721, 38) | cible : 18.19 % de positifs
colonnes catégorielles : ['clinique', 'district', 'sexe', 'quartier', 'assurance', 'rappel_envoye', 'zone_clinique', 'canal_prise']


In [4]:
# 4 — la baseline naïve : « personne ne manque son rendez-vous »
baseline = 100 * max(y.mean(), 1 - y.mean())
print(f"baseline majoritaire : {baseline:.2f} % d'accuracy")

baseline majoritaire : 81.81 % d'accuracy


In [5]:
# 5 — entraîner sur une partie, mesurer sur l'autre (le score INTERNE)
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
modele = HistGradientBoostingClassifier(random_state=0)
modele.fit(Xtr, ytr)
print("score interne :", round(100 * modele.score(Xva, yva), 2), "%")

score interne : 82.77 %


In [6]:
# 6 — prédire sur le test et écrire le fichier de soumission
Xt = test.drop(columns=[c for c in A_ECARTER + TROP_FIN if c in test.columns])
for c in CAT:                       # MÊMES catégories que l'entraînement, sinon décalage silencieux
    Xt[c] = Xt[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)
print(soumission.shape)
print(soumission.head())

(63721, 2)
   ligne_id  patient_absent
0    127994               0
1    188718               0
2     33102               0
3    102132               0
4     81358               0


---

## Et maintenant

1. Cliquez sur **Submit** (à droite) et déposez `submission.csv`.
2. Notez les **trois scores** : baseline naïve, score interne, score public.
3. Regardez l'écart entre votre score interne et votre score public. Il est **moche**.
   C'est normal — et c'est le sujet de la séance 3.

> **Ce que ce notebook ne fait pas :** il ne vous dit pas *pourquoi* ça marche. Il ne vous
> explique ni le gradient, ni la fonction de coût, ni ce que « boosting » veut dire.
> Ces questions vont dans la colonne **Questions en attente** — on y répondra en séance 7.

**Livrable à déposer :** le lien de ce notebook exécuté + votre tableau de 3 scores.